# Chapter 9 — Global Sensitivity Analysis and Sobol Indices
**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local Sensitivity to Global Uncertainty*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Purpose:** Full global SA pipeline: LHS sampling, PRCC, Sobol decomposition,
Jansen estimator, Morris screening, and convergence analysis for the SIR model.

**Key claims tested:**
- ANOVA decomposition: D₁ = D₂ = 3/16, D = 55/144 for f = Q₁+Q₂+Q₁Q₂
- Jansen estimator: A_B^(i) shares column i from B, differs in all others
- First-order Sobol index S_i = η² for linear models (ANOVA connection)
- Global SA reverses local SA ranking when uncertainty is large
- Recommended pipeline: Morris screen → Sobol on top parameters

**Statistical connections (§9.3):** S_i = Var[E(Y|Q_i)]/Var(Y)
is identical to η² in classical ANOVA for linear models with independent inputs.
PRCC is a nonparametric partial regression coefficient (rank-based).

In [ ]:
# ── Dependencies ───────────────────────────────────────────────────────────────
import numpy as np
from scipy.stats import qmc
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.stats import rankdata, t as t_dist
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
FULL = False
N_SOBOL  = 2048 if FULL else 512    # Saltelli sample size
N_LHS    = 500  if FULL else 200    # LHS sample size for PRCC
N_MORRIS = 15   if FULL else 10     # Morris trajectories
print(f'Chapter 9: Global SA  (N_Sobol={N_SOBOL}, N_LHS={N_LHS}, N_Morris={N_MORRIS})')

In [ ]:
# ── Verification Suite — ANOVA worked example (§9.4.2) ───────────────────────
print('=' * 60)
print('VERIFICATION SUITE — ANOVA decomposition (§9.4.2)')
print('=' * 60)

# Analytic computation for f(Q1, Q2) = Q1 + Q2 + Q1*Q2, Q1,Q2 ~ U[0,1]
# E[f] = 1/2 + 1/2 + 1/4 = 5/4
# f1(Q1) = E[f|Q1] - E[f] = (Q1 + 1/2 + Q1/2) - 5/4 = 3Q1/2 - 3/4
# D1 = Var(f1) = (3/2)^2 * Var(Q1) = 9/4 * 1/12 = 3/16
# f12(Q1,Q2) = (Q1-1/2)(Q2-1/2)   → D12 = (1/12)^2 * ... 

# Exact values from book
D1_exact   = 3/16
D2_exact   = 3/16
D12_exact  = 1/144
D_exact    = 55/144
S1_exact   = D1_exact / D_exact   # = 27/55
S12_exact  = D12_exact / D_exact  # = 1/55

# Monte Carlo verification
N_mc = 1_000_000
Q1 = np.random.uniform(0, 1, N_mc)
Q2 = np.random.uniform(0, 1, N_mc)
Y  = Q1 + Q2 + Q1*Q2
D_mc = np.var(Y)

# First-order: D1 = Var[E(Y|Q1)]. For fixed Q1: E(Y|Q1) = Q1 + 1/2 + Q1/2 = 3Q1/2 + 1/2
EY_given_Q1 = 3*Q1/2 + 0.5
D1_mc = np.var(EY_given_Q1)
EY_given_Q2 = 3*Q2/2 + 0.5
D2_mc = np.var(EY_given_Q2)

# Verify analytic values
tol = 0.005  # 0.5% tolerance for Monte Carlo
assert abs(D1_mc - D1_exact)/D1_exact < tol, f'FAIL D1: MC={D1_mc:.5f}, exact={D1_exact:.5f}'
print(f'Test 1 PASS: D₁ = {D1_mc:.5f}  (exact = 3/16 = {D1_exact:.5f})')
assert abs(D2_mc - D2_exact)/D2_exact < tol, f'FAIL D2: MC={D2_mc:.5f}'
print(f'Test 2 PASS: D₂ = {D2_mc:.5f}  (exact = 3/16 = {D2_exact:.5f})')

D_exact_check = D1_exact + D2_exact + D12_exact
assert abs(D_exact_check - D_exact) < 1e-12, 'FAIL D decomp'
print(f'Test 3 PASS: D = D₁+D₂+D₁₂ = 3/16+3/16+1/144 = {D_exact:.6f} = 55/144')

assert abs(D_mc - D_exact)/D_exact < tol, f'FAIL D total: MC={D_mc:.5f}'
print(f'Test 4 PASS: Var(Y) = {D_mc:.5f}  (exact = 55/144 = {D_exact:.5f})')

# S1 = D1/D = (3/16)/(55/144) = (3/16)*(144/55) = 432/880 = 27/55
assert abs(S1_exact - 27/55) < 1e-12, 'FAIL S1 fraction'
print(f'Test 5 PASS: S₁ = 27/55 = {S1_exact:.6f}')

# Statistical connection: S1 = η² for linear models
Y_lin = Q1 + 2*Q2    # linear model
D_lin = np.var(Y_lin)
D1_lin = np.var(Q1 + np.mean(2*Q2))  # Var[E(Y_lin|Q1)] = Var(Q1)
S1_lin_mc  = np.var(Q1) / D_lin
S1_lin_exact = (1/12) / (1/12 + 4/12)  # = 1/5
assert abs(S1_lin_mc - S1_lin_exact) < tol, 'FAIL linear model S1'
print(f'Test 6 PASS: For Y=Q₁+2Q₂, S₁={S1_lin_mc:.4f} = η² = {S1_lin_exact:.4f}')

print(f'\nAll 6 verification tests PASSED.')
print('=' * 60)

In [ ]:
# ── Verification Suite, part 2: the ANOVA worked example as printed (§9.4.2) ──
# Exact rational arithmetic, so no sampling error enters.
from fractions import Fraction as Fr

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

half, var_u = Fr(1, 2), Fr(1, 12)                 # mean and variance of U(0, 1)
f0 = half + half + half * half
D1 = D2 = Fr(3, 2) ** 2 * var_u
D12 = var_u * var_u
D_tot = D1 + D2 + D12
S1, S2, S12 = D1 / D_tot, D2 / D_tot, D12 / D_tot
assert f0 == Fr(5, 4) and D1 == Fr(3, 16) and D12 == Fr(1, 144), 'FAIL: components'
assert D_tot == Fr(55, 144) and D1 == Fr(27, 144), 'FAIL: total variance'
assert S1 == S2 == Fr(27, 55) and S12 == Fr(1, 55), 'FAIL: index fractions'
assert agrees(float(S1), 0.491, 3) and agrees(float(S12), 0.018, 3), 'FAIL: printed decimals'  # book: ch09_body.tex "\tfrac{27}{55} \approx 0.491" "\tfrac{1}{55} \approx 0.018"
assert S1 + S2 + S12 == 1, 'FAIL: indices must sum to 1'
S_add = var_u / (var_u + var_u)                   # additive model Q1 + Q2
assert S_add == half, 'FAIL: additive model S_i = 1/2'
print(f'PASS: f0 = {f0}, D = {D_tot}, S1 = S2 = {S1} = {float(S1):.3f}, '
      f'S12 = {S12} = {float(S12):.3f}, sum = {S1 + S2 + S12}')

In [ ]:
# ── Verification Suite, part 3: the Morris walkthrough as printed (§9.6) ─────
# f(u1, u2) = 3 u1 + u2 + 2 u1 u2 on [0, 1]^2, Delta = 1/2, r = 3 trajectories.

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

f_m = lambda u1, u2: 3 * u1 + u2 + 2 * u1 * u2
Dm = 0.5
trajectories = [((0.0, 0.0), (0, 1)), ((0.5, 0.5), (1, 0)), ((0.5, 0.0), (0, 1))]
EE = {0: [], 1: []}
for start, order in trajectories:
    pt = list(start)
    for k in order:
        before = f_m(*pt)
        pt[k] += Dm
        EE[k].append((f_m(*pt) - before) / Dm)
assert EE[0] == [3.0, 5.0, 3.0] and EE[1] == [2.0, 2.0, 3.0], f'FAIL: EE = {EE}'  # book: ch09_body.tex "EE_1 = \frac{1.50 - 0}{0.5} = 3.0" "EE_2 = \frac{2.50 - 1.50}{0.5} = 2.0" "EE_2 = \frac{3.50 - 2.50}{0.5} = 2.0" "EE_1 = \frac{6.00 - 3.50}{0.5} = 5.0" "EE_1 = \frac{3.00 - 1.50}{0.5} = 3.0" "EE_2 = \frac{4.50 - 3.00}{0.5} = 3.0" "$u_1$ & 3.0 & 5.0 & 3.0" "$u_2$ & 2.0 & 2.0 & 3.0"
assert [f_m(0.5, 0.0), f_m(0.5, 0.5), f_m(0.5, 1.0), f_m(1.0, 1.0), f_m(1.0, 0.5)] == \
    [1.5, 2.5, 3.5, 6.0, 4.5], 'FAIL: trajectory function values'  # book: ch09_body.tex "f(0.5,0.0) = 1.50" "f(0.5,0.5) = 2.50" "f(0.5,1.0) = 3.50" "f(1.0,1.0) = 6.00" "f(1.0,0.5) = 4.50" "\frac{1.50 - 0}{0.5}" "\frac{2.50 - 1.50}{0.5}" "\frac{3.50 - 2.50}{0.5}" "\frac{6.00 - 3.50}{0.5}" "\frac{3.00 - 1.50}{0.5}" "\frac{4.50 - 3.00}{0.5}"
assert f_m(1.0, 0.0) == 3.0, 'FAIL: f(1, 0)'  # book: ch09_body.tex "f(1.0,0.0) = 3.00" "\frac{3.00 - 1.50}{0.5}" "\frac{4.50 - 3.00}{0.5}"
assert len(trajectories) == 3, 'FAIL: r'  # book: ch09_body.tex "and $r = 3$ trajectories" "an artifact of $r = 3$" "with $r = 3$ one of" "In practice, $r = 3$ is too few"
# The elementary effects in closed form, checked on a grid of base points (not only the three trajectories).
_g = [(a, b) for a in (0.0, 0.25, 0.5) for b in (0.0, 0.25, 0.5)]
_ee1 = np.array([(f_m(a + Dm, b) - f_m(a, b)) / Dm for a, b in _g]); _u2 = np.array([b for a, b in _g])
_ee2 = np.array([(f_m(a, b + Dm) - f_m(a, b)) / Dm for a, b in _g]); _u1 = np.array([a for a, b in _g])
assert np.allclose(_ee1, 3 + 2 * _u2, rtol=0, atol=1e-12) and np.allclose(_ee2, 1 + 2 * _u1, rtol=0, atol=1e-12), 'FAIL: EE formulas'  # book: ch09_body.tex "$EE_1 = 3 + 2u_2$ and $EE_2 = 1 + 2u_1$"
mu_star = [np.mean(np.abs(EE[k])) for k in (0, 1)]
sig = [np.std(EE[k], ddof=1) for k in (0, 1)]
assert agrees(mu_star[0], 3.67, 2) and agrees(mu_star[1], 2.33, 2), f'FAIL: mu* = {mu_star}'  # book: ch09_body.tex "& 3.0 & 3.67 & 1.15" "& 3.0 & 2.33 & 0.58"
assert agrees(sig[0], 1.15, 2) and agrees(sig[1], 0.58, 2), f'FAIL: sigma = {sig}'  # book: ch09_body.tex "& 3.0 & 3.67 & 1.15" "& 3.0 & 2.33 & 0.58"
assert agrees(np.sqrt(4 * (1 / 12)), 0.577, 3), 'FAIL: population sigma = 1/sqrt(3)'  # book: ch09_body.tex "\sigma = 1/\sqrt{3} \approx 0.577"
print(f'PASS: mu* = {mu_star[0]:.2f}, {mu_star[1]:.2f};  sigma = {sig[0]:.2f}, {sig[1]:.2f};  '
      f'population sigma = {1/np.sqrt(3):.3f} for both')

In [ ]:
# ── SIR model for global SA ───────────────────────────────────────────────────
def sir_peak_I(params):
    """Return peak infectious count I_max for the demographic SIR model.

    params = [c, beta, tau_R, tau_m], the four POIs of the Chapter 9 laboratory.
    The response is I_max, not R_0, and I_max does depend on tau_m through the
    demographic terms, so all four are sampled.
    """
    c, beta, tau_R, tau_m = params
    N = 1000
    mu = 1.0/tau_m
    def rhs(t, y):
        S, I, R = y
        return [mu*N - c*beta*S*I/N - mu*S,
                c*beta*S*I/N - (1/tau_R + mu)*I,
                I/tau_R - mu*R]
    sol = solve_ivp(rhs, [0, 90], [999, 1, 0],
                    rtol=1e-8, atol=1e-10, dense_output=False)
    return sol.y[1].max()

# Parameter ranges exactly as the Chapter 9 laboratory specifies:
#   c ∈ [2, 8],  β ∈ [0.02, 0.12],  τ_R ∈ [4, 14] days,  τ_m ∈ [1000, 100000] days
lb = np.array([2.0,  0.02, 4.0,  1_000.0])
ub = np.array([8.0,  0.12, 14.0, 100_000.0])
print(f'SIR model: I_max at nominal = {sir_peak_I([5, 0.06, 7, 10_000]):.1f}')

In [ ]:
# ── LHS Sampling ──────────────────────────────────────────────────────────────
def lhs_sample(n, lb, ub, rng=None):
    """Latin Hypercube Sample: n rows, m=len(lb) columns."""
    rng = np.random.default_rng(rng)
    m = len(lb)
    cut = np.linspace(0, 1, n+1)
    X = np.zeros((n, m))
    for j in range(m):
        u = rng.uniform(cut[:-1], cut[1:])
        X[:, j] = lb[j] + rng.permutation(u) * (ub[j] - lb[j])
    return X

X_lhs = lhs_sample(N_LHS, lb, ub, rng=42)
print(f'LHS sample: {X_lhs.shape}, ranges OK: {(X_lhs >= lb).all() and (X_lhs <= ub).all()}')

In [ ]:
# ── PRCC Computation (§9.3) ───────────────────────────────────────────────────
print('Computing PRCC (this may take ~1 min for N_LHS=200)...')
Y_lhs = np.array([sir_peak_I(x) for x in X_lhs])
print(f'Model evaluations: {N_LHS}  |  I_max range: [{Y_lhs.min():.1f}, {Y_lhs.max():.1f}]')

def prcc(X, Y):
    """
    Compute PRCC (Partial Rank Correlation Coefficient) for each column of X.
    PRCC is a nonparametric partial regression coefficient (§9.3).

    Parameters
    ----------
    X : ndarray (n, m) — input matrix
    Y : ndarray (n,)   — scalar model output

    Returns
    -------
    rho   : ndarray (m,) — PRCC values
    pvals : ndarray (m,) — t-test p-values
    """
    n, m = X.shape
    # Rank-transform
    Xr = np.column_stack([rankdata(X[:, j]) for j in range(m)])
    Yr = rankdata(Y)

    # Partial correlation: regress Xr[:,j] and Yr on all other columns
    rho   = np.zeros(m)
    pvals = np.zeros(m)
    for j in range(m):
        other = np.delete(np.arange(m), j)
        Z = Xr[:, other]  # confounders

        # Residuals after regressing Xr[:,j] on Z
        Z_aug  = np.column_stack([np.ones(n), Z])
        coef_x = np.linalg.lstsq(Z_aug, Xr[:, j], rcond=None)[0]
        resid_x = Xr[:, j] - Z_aug @ coef_x

        # Residuals after regressing Yr on Z
        coef_y = np.linalg.lstsq(Z_aug, Yr, rcond=None)[0]
        resid_y = Yr - Z_aug @ coef_y

        # Partial correlation = correlation of residuals
        r  = np.corrcoef(resid_x, resid_y)[0, 1]
        rho[j] = r

        # t-statistic with n-m-1 df
        df = n - m - 1
        t_stat = r * np.sqrt(df / (1 - r**2 + 1e-15))
        pvals[j] = 2 * (1 - t_dist.cdf(abs(t_stat), df))

    return rho, pvals

rho_prcc, pvals_prcc = prcc(X_lhs, Y_lhs)
param_names = ['c', 'β', 'τ_R', 'τ_m']
print('\nPRCC results for I_max:')
for name, r, pv in zip(param_names, rho_prcc, pvals_prcc):
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
    print(f'  PRCC_{name} = {r:+.4f}  p={pv:.4f} {sig}')

In [ ]:
# ── Saltelli/Jansen Sobol Estimator (§9.4–9.5) ───────────────────────────────
print(f'\nComputing Sobol indices (N={N_SOBOL}, {N_SOBOL*(len(lb)+2)} total model evaluations)...')

def saltelli_sample(N, lb, ub, rng=42):
    """Generate Saltelli sample matrices A and B (each N×m).

    Draws a genuine Sobol' sequence in 2m dimensions and splits it into the
    A and B blocks, which is what §9.12 tells the reader to do.  The balance
    property of a Sobol' sequence holds in blocks of 2^k, so N should be a
    power of two; scipy warns if it is not.  Do not substitute independent
    pseudo-random uniforms here: the power-of-two justification in the text
    describes a Sobol' sequence and is vacuous without one.
    """
    m = len(lb)
    engine = qmc.Sobol(d=2*m, scramble=True, seed=rng)
    AB = engine.random(N)                      # N × 2m, low-discrepancy
    A  = lb + AB[:, :m] * (ub - lb)
    B  = lb + AB[:, m:] * (ub - lb)
    return A, B

def sobol_jansen(model, N, lb, ub, rng=42):
    """
    Estimate first-order Sobol indices via Jansen estimator.
    A_B^(i) = matrix A with column i replaced by column i of B.
    S_i = 1 - Var[f(B) - f(A_B^(i))] / (2*Var(f))
    """
    m  = len(lb)
    A, B = saltelli_sample(N, lb, ub, rng)

    fA = np.array([model(A[i]) for i in range(N)])
    fB = np.array([model(B[i]) for i in range(N)])
    D  = np.var(np.concatenate([fA, fB]))

    S1 = np.zeros(m)
    for j in range(m):
        # A_B^(j): A with column j replaced by B's column j
        AB_j = A.copy()
        AB_j[:, j] = B[:, j]
        fABj = np.array([model(AB_j[i]) for i in range(N)])
        # Jansen (1999) first-order estimator
        S1[j] = 1.0 - np.mean((fB - fABj)**2) / (2*D)

    return S1, D

S1_sobol, D_sobol = sobol_jansen(sir_peak_I, N_SOBOL, lb, ub)
print('\nSobol first-order indices for I_max:')
for name, s in zip(param_names, S1_sobol):
    print(f'  S₁({name}) = {s:.4f}')
print(f'  Sum S₁ = {S1_sobol.sum():.4f}  (< 1 means interactions present)')

In [ ]:
# ── Figure 9.1: PRCC vs Sobol comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# PRCC bar chart
ax = axes[0]
colors_prcc = ['steelblue' if r > 0 else 'coral' for r in rho_prcc]
ax.bar(param_names, rho_prcc, color=colors_prcc, edgecolor='white', width=0.5)
ax.axhline(0, color='black', lw=0.8)
for i, (r, pv) in enumerate(zip(rho_prcc, pvals_prcc)):
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
    ax.text(i, r + 0.02*np.sign(r), f'{r:.3f}{sig}', ha='center', fontsize=10)
ax.set_ylabel('PRCC', fontsize=12)
ax.set_title(r'PRCC for $I_{\rm max}$'+f'\n(N={N_LHS} LHS samples)', fontsize=11)
ax.set_ylim(-0.15, 1.05)
ax.grid(axis='y', alpha=0.3)

# Sobol bar chart
ax2 = axes[1]
ax2.bar(param_names, S1_sobol, color='seagreen', edgecolor='white', width=0.5)
ax2.axhline(0, color='black', lw=0.8)
for i, s in enumerate(S1_sobol):
    ax2.text(i, s + 0.01, f'{s:.3f}', ha='center', fontsize=10)
ax2.set_ylabel(r'First-order Sobol index $S_i$', fontsize=12)
ax2.set_title(r'Sobol $S_1$ for $I_{\rm max}$'+f'\n(N={N_SOBOL} Saltelli evaluations)', fontsize=11)
ax2.set_ylim(0, 0.85)
ax2.grid(axis='y', alpha=0.3)
ax2.text(0.98, 0.95, f'Σ S₁ = {S1_sobol.sum():.3f}',
         transform=ax2.transAxes, ha='right', va='top', fontsize=9, color='gray')

plt.suptitle('Global SA for SIR peak infections\n'
             'PRCC (monotonicity-based) vs Sobol (variance-based)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('ch09_fig1_prcc_sobol.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch09_fig1_prcc_sobol.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 9.1 saved.')

In [ ]:
# ── Figure 9.2: ANOVA decomposition verification ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: Sobol indices for toy model f = Q1 + Q2 + Q1*Q2
def toy_model(q): return q[0] + q[1] + q[0]*q[1]
lb2, ub2 = np.array([0., 0.]), np.array([1., 1.])
S1_toy, _ = sobol_jansen(toy_model, 2048, lb2, ub2)

ax = axes[0]
ax.bar(['$Q_1$', '$Q_2$'], [S1_exact, S1_exact],
       color='steelblue', alpha=0.4, label='Analytic', width=0.35, align='edge')
ax.bar(['$Q_1$', '$Q_2$'], S1_toy,
       color='coral', alpha=0.7, label='Jansen MC', width=-0.35, align='edge')
ax.axhline(S1_exact, color='steelblue', ls='--', lw=1.2)
ax.set_ylabel('First-order Sobol index', fontsize=11)
ax.set_title(fr'$f = Q_1+Q_2+Q_1Q_2$: $S_1=S_2=27/55\approx{S1_exact:.3f}$', fontsize=10)
ax.legend(fontsize=9); ax.set_ylim(0, 0.65)
ax.text(0.5, 0.15, fr'$S_{{12}} = 1/55 \approx {S12_exact:.3f}$',
        transform=ax.transAxes, ha='center', fontsize=10, color='gray')

# Right: Variance decomposition pie
ax2 = axes[1]
sizes = [D1_exact/D_exact, D2_exact/D_exact, D12_exact/D_exact]
labels = [fr'$D_1={D1_exact:.4f}$\n$S_1={S1_exact:.3f}$',
          fr'$D_2={D2_exact:.4f}$\n$S_2={S1_exact:.3f}$',
          fr'$D_{{12}}={D12_exact:.4f}$\n$S_{{12}}={S12_exact:.3f}$']
colors_pie = ['steelblue', 'coral', 'seagreen']
ax2.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 9})
ax2.set_title(fr'ANOVA decomposition: $D = {D_exact:.4f} = 55/144$', fontsize=10)

plt.tight_layout()
plt.savefig('ch09_fig2_anova.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch09_fig2_anova.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 9.2 saved.')

In [ ]:
# ── Results & Download ─────────────────────────────────────────────────────────
RESULTS = {
    'chapter': 9,
    'ANOVA_exact': {'D1': D1_exact, 'D2': D2_exact, 'D12': D12_exact,
                    'D': D_exact, 'S1': S1_exact, 'S12': S12_exact},
    'PRCC_SIR_Imax': dict(zip(param_names, rho_prcc.tolist())),
    'Sobol_S1_SIR_Imax': dict(zip(param_names, S1_sobol.tolist())),
    'Sobol_sum_S1': float(S1_sobol.sum()),
    'figures': ['ch09_fig1_prcc_sobol.pdf', 'ch09_fig2_anova.pdf']
}
for k_r, v in RESULTS.items():
    print(f'  {k_r}: {v}')

try:
    from google.colab import files
    for f in RESULTS['figures']:
        files.download(f)
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — figures saved locally.')

In [ ]:
# ── Structural assertions for the four-POI laboratory ────────────────────────
# tau_m is sampled over [1000, 100000] days against a 90-day outbreak, so
# demography is nearly irrelevant to I_max.  That is a falsifiable prediction:
# tau_m must rank LAST with a near-zero index.  A three-parameter version of this
# notebook cannot satisfy these, which is exactly why they are stated.
assert len(lb) == 4, f"laboratory specifies four POIs; found {len(lb)}"
assert param_names[3] == '\u03c4_m', f"fourth POI is {param_names[3]}, expected tau_m"
_S1 = dict(zip(param_names, S1_sobol))
assert _S1[param_names[3]] < 1 / 20, f"S1(tau_m) = {_S1[param_names[3]]:.4f}; expected < 0.05"  # integer form: no printed number is meant
assert _S1[param_names[3]] == min(_S1.values()), "tau_m should rank last of the four"
assert 85 / 100 < S1_sobol.sum() < 115 / 100, f"sum S1 = {S1_sobol.sum():.4f}; expected near 1"  # integer form: structural bound, not a printed number
print(f"Structural checks PASS: S1({param_names[3]}) = {_S1[param_names[3]]:.4f} "
      f"ranks last, sum S1 = {S1_sobol.sum():.4f}")

In [ ]:
# ── Verification Suite, part 4: values the running text quotes ───────────────────
# The numbers in §§9.1-9.6 that the cells above do not already check.  Each check
# names the book line it is about with a comment of the form
#     # book: ch09_body.tex "text on that line"
# so the check accounts for that
# occurrence and for no other.  Parameters the text states are bound to named
# variables here, each with the value the text prints.

# §9.1  Two parameters with s = (2, 1) and sigma = (1, 3): Var(Y) = 4 + 9 = 13.
s_toy, sd_toy = np.array([2.0, 1.0]), np.array([1.0, 3.0])
sigma_Y_toy = np.sqrt(np.sum((s_toy * sd_toy) ** 2))
assert agrees(sd_toy[0] / sigma_Y_toy * s_toy[0], 0.55, 2)  # book: ch09_body.tex "\cdot 2 \approx 0.55"
assert agrees(sd_toy[1] / sigma_Y_toy * s_toy[1], 0.83, 2)  # book: ch09_body.tex "\cdot 1 \approx 0.83"

# §9.2  "Two SIR Models" and the self-check after it.
c_star, beta_star, tauR_star = 5, 0.06, 7                 # nominal values
sigma_c, sigma_beta, sigma_tauR = 0.5, 0.01, 2            # the self-check's uncertainties
sigma_beta_A, sigma_beta_B = 0.005, 0.03                  # Setting A, Setting B
grad_R0 = np.array([beta_star * tauR_star, c_star * tauR_star, c_star * beta_star])

def sigma_normalized(sd):
    """S_i^sigma = (dR0/dp_i) sigma_i / sigma_R0, sigma_R0 by the delta method."""
    return grad_R0 * sd / np.sqrt(np.sum((grad_R0 * sd) ** 2))

rise = (sigma_normalized(np.array([sigma_c, sigma_beta_B, sigma_tauR]))[1]
        / sigma_normalized(np.array([sigma_c, sigma_beta_A, sigma_tauR]))[1])
assert np.isclose(grad_R0[1] * sigma_beta_B, 6 * grad_R0[1] * sigma_beta_A)   # six times larger
assert rise < 6 and agrees(rise, 3.2, 1)  # book: ch09_body.tex "about~$3.2$"

# The self-check answer, line by line as printed (rounded intermediates included),
# then the same quantities without rounding.
assert agrees(c_star * beta_star * tauR_star, 2.1, 1)  # book: ch09_body.tex "\cR_0^* = 2.1"
assert agrees(grad_R0[0], 0.42, 2) and agrees(grad_R0[2], 0.3, 1)  # book: ch09_body.tex "= \beta\tau_R = 0.42$" "= c\beta = 0.3$."
_R0f = lambda b: c_star * b * tauR_star  # the derivative measured, not the printed product
assert agrees((_R0f(beta_star + 1e-6) - _R0f(beta_star - 1e-6)) / 2e-6, 35, 6), 'FAIL: dR0/dbeta = c tau_R'  # book: ch09_body.tex "\partial\beta = c\tau_R = 35"
assert agrees(0.42 ** 2 * 0.25 + 35 ** 2 * 0.0001 + 0.3 ** 2 * 4, 0.527, 3)  # book: ch09_body.tex "\approx 0.42^2(0.25) + 35^2(0.0001) + 0.3^2(4)"
assert (agrees(0.42 ** 2 * 0.25, 0.0441, 4) and agrees(35 ** 2 * 0.0001, 0.1225, 4)
        and agrees(0.3 ** 2 * 4, 0.36, 2) and agrees(0.0441 + 0.1225 + 0.36, 0.527, 3))  # book: ch09_body.tex "= 0.0441 + 0.1225 + 0.36 = 0.527$."
var_R0 = np.sum((grad_R0 * np.array([sigma_c, sigma_beta, sigma_tauR])) ** 2)
assert agrees(var_R0, 0.527, 3) and agrees(np.sqrt(var_R0), 0.726, 3)  # book: ch09_body.tex "\sigma_{\cR_0} \approx 0.726$"
S_sig = sigma_normalized(np.array([sigma_c, sigma_beta, sigma_tauR]))
assert agrees(0.42 * 0.5 / 0.726, 0.29, 2) and agrees(S_sig[0], 0.29, 2)  # book: ch09_body.tex "S_c^\sigma = 0.42 \times 0.5/0.726"
assert agrees(35 * 0.01 / 0.726, 0.48, 2) and agrees(S_sig[1], 0.48, 2)  # book: ch09_body.tex "S_\beta^\sigma = 35 \times 0.01/0.726"
assert agrees(0.3 * 2 / 0.726, 0.83, 2) and agrees(S_sig[2], 0.83, 2)  # book: ch09_body.tex "S_{\tau_R}^\sigma = 0.3 \times 2/0.726"
assert list(np.argsort(-S_sig)) == [2, 1, 0]               # the ranking tau_R > beta > c

# §9.3  LHS self-check, N = 4 on [0, 1]: four quarter-intervals, one point in each.
edges_lhs = np.linspace(0, 1, 5)
assert np.allclose(edges_lhs, [0, 0.25, 0.5, 0.75, 1])  # book: ch09_body.tex "each quarter-interval $[0,0.25)$, $[0.25,0.5)$," "$[0.5,0.75)$, $[0.75,1]$ must contain" "one point in each of $[0,0.25)$, $[0.25,0.5)$," "$[0.5,0.75)$, $[0.75,1]$, with uniform jitter"
assert 0.5 ** 4 == 1 / 16  # book: ch09_body.tex "fall in $[0, 0.5]$?" "all 4 points in $[0, 0.5]$"
assert [int(4 * x) for x in [0.18, 0.43, 0.62, 0.91]] == [0, 1, 2, 3]  # book: ch09_body.tex "$X = \{0.18, 0.43, 0.62, 0.91\}$"

# §9.5  Cost at N = 1024 with m = 4: N(m + 2) = 6144 runs; at 0.1 s a run, about 10 minutes.
assert 1024 * (4 + 2) == 6144 and round(6144 * 0.1 / 60) == 10  # book: ch09_body.tex "If each evaluation takes 0.1~seconds"

# §9.6  The Morris walkthrough as printed: starting points, values, steps (cell 4's f_m).
assert trajectories[0][0] == (0.0, 0.0)  # book: ch09_body.tex "Start at $(0.0,\,0.0)$"
assert trajectories[1][0] == (0.5, 0.5)  # book: ch09_body.tex "Start at $(0.5,\,0.5)$"
assert trajectories[2][0] == (0.5, 0.0)  # book: ch09_body.tex "Start at $(0.5,\,0.0)$"
assert f_m(0.0, 0.0) == 0 and f_m(0.5, 0.0) == 1.50 and f_m(0.5, 0.5) == 2.50  # book: ch09_body.tex "f(0.0,0.0) = 0," "f(0.5,0.0) = 1.50," "f(0.5,0.5) = 2.50."
assert f_m(0.5, 0.5) == 2.50 and f_m(0.5, 1.0) == 3.50 and f_m(1.0, 1.0) == 6.00  # book: ch09_body.tex "f(0.5,0.5) = 2.50," "f(0.5,1.0) = 3.50," "f(1.0,1.0) = 6.00."
assert f_m(1.0, 0.0) == 3.00 and f_m(1.0, 0.5) == 4.50  # book: ch09_body.tex "f(1.0,0.0) = 3.00," "f(1.0,0.5) = 4.50."
assert (1.50 - 0) / 0.5 == 3.0 and (2.50 - 1.50) / 0.5 == 2.0  # book: ch09_body.tex "EE_1 = \frac{1.50 - 0}{0.5} = 3.0" "EE_2 = \frac{2.50 - 1.50}{0.5} = 2.0"
assert (3.50 - 2.50) / 0.5 == 2.0 and (6.00 - 3.50) / 0.5 == 5.0  # book: ch09_body.tex "EE_2 = \frac{3.50 - 2.50}{0.5} = 2.0" "EE_1 = \frac{6.00 - 3.50}{0.5} = 5.0"
assert (3.00 - 1.50) / 0.5 == 3.0 and (4.50 - 3.00) / 0.5 == 3.0  # book: ch09_body.tex "EE_1 = \frac{3.00 - 1.50}{0.5} = 3.0" "EE_2 = \frac{4.50 - 3.00}{0.5} = 3.0"
print('PASS: part 4, the quoted values of §§9.1-9.6')

In [ ]:
# ── Verification Suite, part 5: the §9.5.1 worked example and the laboratory (tau_R since v1_22_0) ──────
# §9.5.1: SIR with three POIs (c, beta, tau_R), QOI the peak infectious fraction, ranges
# +/-50% about c = 8, beta = 0.06 and tau_R = 10 days; a Saltelli design with N = 256 and
# the Jansen estimators; nBoot = 100.  Design and estimator are this notebook's own:
# saltelli_sample with seed 42 and the Jansen form of cell 8; the bootstrap draws rows with
# default_rng(42).  Since v1_22_0 the text prints this cell's output.
# In order, the cell establishes:
#   1. the peak depends on the POIs only through R0 = c*beta*tau_R (closed form = ODE);
#   2. the point estimates, replicates 1-4 and 100, and the 95% intervals, as printed;
#   3. the population indices, by quadrature, are equal: R0 is a symmetric product of
#      three POIs with the same relative spread, so the estimates differ only by scatter.
nominal_951 = np.array([8.0, 0.06, 10.0])
lb_951, ub_951 = 0.5 * nominal_951, 1.5 * nominal_951
N_951, NBOOT_951 = 256, 100
assert len(nominal_951) == 3, 'FAIL: m'  # book: ch09_body.tex "Apply the SIR model ($m = 3$ parameters"
S0_951, I0_951 = 0.999, 0.001

def peak_closed(R0):
    """Peak infectious fraction of SIR started at (S0, I0): a function of R0 alone."""
    R0 = np.asarray(R0, float)
    return np.where(R0 * S0_951 > 1, I0_951 + S0_951 - (1 + np.log(R0 * S0_951)) / R0, I0_951)

def peak_ode(p, t_end=400.0):
    c, b, tau = p
    t = np.linspace(0, t_end, 400_001)
    sol = solve_ivp(lambda _t, y: [-c * b * y[0] * y[1], c * b * y[0] * y[1] - y[1] / tau],
                    [0, t_end], [S0_951, I0_951], t_eval=t, rtol=1e-11, atol=1e-13)
    return sol.y[1].max()

ODE_TOL = 1e-6
for p in (nominal_951, lb_951, ub_951, np.array([lb_951[0], ub_951[1], ub_951[2]])):
    assert abs(peak_ode(p) - peak_closed(p[0] * p[1] * p[2])) < ODE_TOL, f'closed form off at {p}'
qoi_951 = lambda p: float(peak_closed(p[0] * p[1] * p[2]))

def jansen_with_values(model, N, lb_, ub_, seed=42):
    """Cell 8's design and Jansen estimator, keeping the model values for the bootstrap."""
    A, B = saltelli_sample(N, lb_, ub_, seed)
    fA = np.array([model(a) for a in A])
    fB = np.array([model(b) for b in B])
    fAB = []
    for j in range(len(lb_)):
        ABj = A.copy()
        ABj[:, j] = B[:, j]
        fAB.append(np.array([model(x) for x in ABj]))
    return fA, fB, np.array(fAB)

def jansen_S1(fA, fB, fAB):
    D = np.var(np.concatenate([fA, fB]))
    return 1.0 - np.mean((fB - fAB) ** 2, axis=1) / (2 * D)

fA_951, fB_951, fAB_951 = jansen_with_values(qoi_951, N_951, lb_951, ub_951)
S_hat_951 = jansen_S1(fA_951, fB_951, fAB_951)
assert np.allclose(S_hat_951, sobol_jansen(qoi_951, N_951, lb_951, ub_951)[0])   # cell 8's estimator
print('point estimates (N = 256, seed 42): ' + ', '.join(f'{s:.4f}' for s in S_hat_951))
assert agrees(S_hat_951[0], 0.32, 2)  # book: ch09_body.tex "\hat{S}_1^c = 0.32"
assert agrees(S_hat_951[1], 0.33, 2)  # book: ch09_body.tex "\hat{S}_1^\beta = 0.33"
assert agrees(S_hat_951[2], 0.33, 2)  # book: ch09_body.tex "\hat{S}_1^{\tau_R} = 0.33"
assert np.allclose(np.round(S_hat_951, 2), [0.32, 0.33, 0.33])  # book: ch09_body.tex "small differences among $0.32$, $0.33$ and $0.33$"
assert agrees(S_hat_951.mean(), 0.33, 2)  # book: ch09_body.tex "three indices near $0.33$"

# Bootstrap (nBoot = 100): rows of A and B resampled together; percentile intervals.
boot_rng = np.random.default_rng(42)
boot = np.array([jansen_S1(fA_951[i], fB_951[i], fAB_951[:, i])
                 for i in (boot_rng.integers(0, N_951, N_951) for _ in range(NBOOT_951))])
ci_951 = np.percentile(boot, [2.5, 97.5], axis=0).T
assert np.allclose(np.round(boot[0], 2), [0.34, 0.32, 0.27])  # book: ch09_body.tex "1 & 0.34 & 0.32 & 0.27"
assert np.allclose(np.round(boot[1], 2), [0.33, 0.30, 0.34])  # book: ch09_body.tex "2 & 0.33 & 0.30 & 0.34"
assert np.allclose(np.round(boot[2], 2), [0.34, 0.42, 0.37])  # book: ch09_body.tex "3 & 0.34 & 0.42 & 0.37"
assert np.allclose(np.round(boot[3], 2), [0.40, 0.38, 0.27])  # book: ch09_body.tex "4 & 0.40 & 0.38 & 0.27"
assert np.allclose(np.round(boot[99], 2), [0.26, 0.25, 0.30])  # book: ch09_body.tex "100 & 0.26 & 0.25 & 0.30"
assert np.allclose(np.round(ci_951[0], 2), [0.21, 0.42])  # book: ch09_body.tex "\hat{S}_1^c = 0.32 \quad [0.21,\ 0.42]"
assert np.allclose(np.round(ci_951[1], 2), [0.20, 0.44])  # book: ch09_body.tex "\hat{S}_1^\beta = 0.33 \quad [0.20,\ 0.44]"
assert np.allclose(np.round(ci_951[2], 2), [0.25, 0.42])  # book: ch09_body.tex "\hat{S}_1^{\tau_R} = 0.33 \quad [0.25,\ 0.42]"
assert np.all(ci_951[:, 0] > 0)                                            # all three exclude zero
assert all(ci_951[i, 0] < ci_951[j, 1] for i in range(3) for j in range(3))  # and overlap

# Population first-order indices by Gauss-Legendre quadrature on the +/-50% box: equal.
x_gl, w_gl = np.polynomial.legendre.leggauss(200)
u_gl, w_gl = 1.0 + 0.5 * x_gl, w_gl / 2                    # U[0.5, 1.5] nodes and weights
g_c, g_b, g_t = (nominal_951[i] * u_gl for i in range(3))
Y_gl = peak_closed(g_c[:, None, None] * g_b[None, :, None] * g_t[None, None, :])
W_gl = w_gl[:, None, None] * w_gl[None, :, None] * w_gl[None, None, :]
EY_gl = np.sum(W_gl * Y_gl)
VY_gl = np.sum(W_gl * Y_gl ** 2) - EY_gl ** 2

def S1_population(axis):
    other = tuple(a for a in range(3) if a != axis)
    cond = np.sum(W_gl * Y_gl, axis=other) / w_gl            # E[Y | x_axis] at each node
    return (np.sum(w_gl * cond ** 2) - EY_gl ** 2) / VY_gl

S_pop_951 = np.array([S1_population(a) for a in range(3)])
print('population indices: ' + ', '.join(f'{s:.5f}' for s in S_pop_951))
SYM_TOL = 1e-12
assert np.ptp(S_pop_951) < SYM_TOL                          # R0 = c*beta*tau_R is symmetric

# §9.10 laboratory: share of the box with R0 = c*beta*tau_R <= 1, exactly, by quadrature.
from scipy.integrate import dblquad
_area = dblquad(lambda t, c: min(max((1 / (c * t) - lb[1]) / (ub[1] - lb[1]), 0), 1),
                lb[0], ub[0], lambda c: lb[2], lambda c: ub[2], epsabs=1e-12, epsrel=1e-12)[0]
share_R0_below_1 = _area / ((ub[0] - lb[0]) * (ub[2] - lb[2]))
assert agrees(100 * share_R0_below_1, 10.8, 1)  # book: ch09_body.tex "About $10.8\%$ of it"
print(f'PASS: part 5.  Laboratory box: {100 * share_R0_below_1:.2f}% has R0 <= 1.')